# Random walks on the cube: Python and C++ in one experiment

**Study setup.** I keep the mathematical transition rules the same in both languages while changing how their computation is implemented. For a selected positive integer dimension \(n\), each of the five chains starts at the origin in \([-1/2,1/2]^n\). I save independent endpoints, convergence diagnostics and the runtime of each implementation. The final comparisons ask separately how many sampled transitions are needed to satisfy an **empirical** criterion and how many seconds of computation they cost.

**Why a Python notebook for C++?** The Python cells *compile and launch* the real C++17 executable and then read its saved CSV output. The C++ walk calculations occur in native compiled C++, not in Python. I do not need an additional C++ kernel.

The full mathematical definitions are in `docs/01_MATHEMATICAL_METHODS.md`, and the optimization changes are listed at the **end** of `README.md`.

In [ ]:
from pathlib import Path
import subprocess
import sys
import csv
from IPython.display import display, Image, Markdown

# Works whether Jupyter starts in the project root or in notebooks/.
HERE = Path.cwd().resolve()
PROJECT = HERE if (HERE / "run_all.py").exists() else HERE.parent
assert (PROJECT / "run_all.py").exists(), "Open this notebook inside the extracted project folder."
print("Project:", PROJECT)
print("Python:", sys.executable)

## 1. Select all parameters in one place

Set `N` to any positive integer. Keep 2560 steps and 3000 chains to reproduce the earlier **n=20** experiment; use smaller numbers initially if checking your installation. `THREADS=1` is a single-thread comparison to the original C++; `THREADS=4` allows the optimized program to run independent chains concurrently.

The seed identifies a reproducible random stream **within one implementation**, not an identical sequence shared between Python and C++.

In [ ]:
N = 20                   # Change to 1, 2, 3, 10, 25, ...
STEPS = 2560              # Markov steps for each independent chain
CHAINS = 3000             # Independent chains starting at zero
SEED = 2026
REPEATS = 1
THREADS = 4               # Select 1 for a single-thread C++ benchmark
SNAPSHOT_CHAINS = 300     # Save just these many chains at each checkpoint

assert N >= 1 and STEPS >= 1 and CHAINS >= 1 and REPEATS >= 1
print(f"Selected: n={N}, steps={STEPS}, chains={CHAINS}, seed={SEED}")

## 2. Execute *both* languages and generate all outputs

Run this cell **once**. It runs the Python and C++ versions, creates a unique dated directory in `runs/`, and writes their numerical CSVs, comparison matrices, 2D/3D/projection plots and runtime charts. This can be a substantial computation at n=20, 3000 chains and 2560 steps. The earlier experiments are retained, rather than overwritten.

In [ ]:
before = set((PROJECT / "runs").iterdir())
command = [
    sys.executable, str(PROJECT / "run_all.py"),
    "--dimensions", str(N),
    "--steps", str(STEPS),
    "--chains", str(CHAINS),
    "--seed", str(SEED),
    "--repeats", str(REPEATS),
    "--threads", str(THREADS),
    "--snapshot-chains", str(SNAPSHOT_CHAINS),
]
subprocess.run(command, cwd=PROJECT, check=True)
after = set((PROJECT / "runs").iterdir())
new_runs = [p for p in (after - before) if p.is_dir()]
assert len(new_runs) == 1, "Run completed but output folder could not be identified uniquely"
RUN_DIRECTORY = new_runs[0]
print("Saved results in:", RUN_DIRECTORY)

## 3. Read the numerical comparison

This is a **summary of a full new experiment**. The `*_proxy_steps` columns are observed checkpoints that pass an empirical threshold at consecutive sampled times; they are **not certified total-variation mixing times**. A blank cell means the threshold was not observed.

In [ ]:
with (RUN_DIRECTORY / "tables" / "combined_results.csv").open(newline="") as f:
    comparison = list(csv.DictReader(f))
print("Walk | n | Python checkpoint | C++ checkpoint | Python µs/step | C++ µs/step")
for r in comparison:
    print(f'{r["walk"]:20s} | {r["n"]:>2} | {r["python_proxy_steps"] or "not reached":>11} | {r["cpp_proxy_steps"] or "not reached":>11} | {r["python_us_per_step"]:>10} | {r["cpp_us_per_step"]:>10}')

## 4. View density, geometry, convergence and runtime

The plotting scripts use Matplotlib **only to draw saved experimental samples**. For n=1 use histograms, n=2 square scatter plots plus 2D densities, n=3 3D clouds, and higher dimensions 2D coordinate projection densities. Plots do not show the full high-dimensional law.

In [ ]:
figures = sorted((RUN_DIRECTORY / "figures").glob("*.png"))
for figure in figures:
    print("Figure:", figure.name)
    display(Image(filename=str(figure), width=900))

## 5. Research interpretation

**Fixed n:** A kernel that passes a chosen empirical criterion in fewer Markov steps is not necessarily the cheapest in wall time. **Changing n:** compare the walk-by-dimension matrices in the output directory, and do not infer an asymptotic exponent from ten sampled dimensions. **Exact control:** Coordinate Hit-and-Run has a known coupon-collector total-variation formula for the origin start; this can reveal discrepancies between full-law mixing and marginal statistics.

The exact settings, time stamp, versions and other reproducibility metadata are preserved in `parameters.json`. For a stable interpretation we should use independent seeds, repeat timings and compare distributions—not demand identical Python/C++ sample paths.

In [ ]:
display(Markdown((RUN_DIRECTORY / "tables" / "HOW_TO_READ_TABLES.md").read_text()))
display(Markdown((PROJECT / "docs" / "03_OPTIMIZATION_CHANGES.md").read_text()))